In [2]:
from typing_extensions import TypedDict
from langchain_google_genai  import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, StateGraph, START
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv

In [6]:
class SharedState(TypedDict):
    query: str
    model: ChatGoogleGenerativeAI
    from_ml_topic: bool
    ai_answer: str

class GraderOutput(TypedDict):
    from_machine_learning_topic: bool

In [7]:
def build_model(shared_state: SharedState):
    model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
    shared_state['model'] = model
    return shared_state

In [18]:
def get_query_topic(shared_state: SharedState):
    print("Determining if the query is related to machine learning topics...")
    
    prompt = """
You are a classifier that determines if a user's query is related to machine learning topics.
Given the user's query, return True if it is related to machine learning topics, otherwise return False.
    """

    model = shared_state['model']
    structured_llm_grader = model.with_structured_output(GraderOutput)
    query = shared_state['query']

    grade_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", prompt),
            ("human", "User's Query: \n\n {query}"),
        ]
    )

    retrieval_grader = grade_prompt | structured_llm_grader

    result = retrieval_grader.invoke({"query": query})
    shared_state['from_ml_topic'] = result['from_machine_learning_topic']

    return shared_state

In [19]:
def grader_node(shared_state: SharedState):
    if shared_state['from_ml_topic']:
        return "continue"

    return "exit"

In [20]:
def answer_query(shared_state: SharedState):
    print("Answering the user's query...")
    prompt = """
You are an expert in machine learning. Answer the user's query under 200 words.
    """
    model = shared_state['model']
    query = shared_state['query']
    answer_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", prompt),
            ("human", "User's Query: \n\n {query}"),
        ]
    )
    answer_chain = answer_prompt | model | StrOutputParser()
    result = answer_chain.invoke({"query": query})
    shared_state['ai_answer'] = result

    return shared_state

In [21]:
def build_graph():
    workflow = StateGraph(SharedState)

    # Add Nodes
    workflow.add_node(build_model, "build_model")
    workflow.add_node(get_query_topic, "get_query_topic")
    workflow.add_node(answer_query, "answer_query")

    workflow.add_edge(START, "build_model")
    workflow.add_edge("build_model", "get_query_topic")
    workflow.add_conditional_edges(
        "get_query_topic", 
        grader_node, 
        { 
            "continue": "answer_query",
            "exit": END 
        }
    )
    workflow.add_edge("answer_query", END)

    return workflow.compile()

In [22]:
# Query 1: "What are the latest advancements in machine learning?"
# Query 2: "What is the capital of India?"
def execute_prompt_chain_workflow():
    workflow = build_graph()
    initial_state: SharedState = {
        "query": "What is the capital of India?",
    }

    agent_response = workflow.invoke(initial_state)
    print(agent_response)

    if agent_response['from_ml_topic']:
        print("AI's Answer:", agent_response['ai_answer'])
    else:
        print("The query is not related to machine learning topics.")

In [23]:
load_dotenv()
execute_prompt_chain_workflow()

Determining if the query is related to machine learning topics...
{'query': 'What is the capital of India?', 'model': ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12', 'langchain-google-genai': '4.2.7'}}, output_version=None, profile={'name': 'Gemini 3 Flash Preview', 'release_date': '2025-12-17', 'last_updated': '2025-12-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3-flash-preview', temperature=1.0, client=<google.genai.client.Client object at 